<a href="https://colab.research.google.com/github/ruan98651-arty/Projeto05_Amazonia-Bank-Mini-Banco-Digital/blob/main/projeto05_Amazoniabank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Projeto 05 - Amazonia Bank : Mini Banco Digital**

In [6]:
# =====================================================================
# Script de Gerenciamento Bancário Interativo - Amazônia Bank (Projeto 05)
# Descrição: Permite cadastrar clientes dinamicamente via terminal (input),
# solicitar senha de 4 dígitos, realizar depósitos, saques e visualizar
# informações de clientes sem dados pré-cadastrados.
# =====================================================================

class Cliente:
    """Armazena as informações pessoais do cliente."""
    def __init__(self, nome, cpf, telefone):
        self.nome = self._validar_nome(nome)
        self.cpf = cpf.strip()
        self.telefone = telefone.strip()

    def _validar_nome(self, nome):
        nome_limpo = nome.strip()
        if not nome_limpo or ";" in nome_limpo:
            raise ValueError("Nome de cliente inválido ou contém caractere proibido (;).")
        return nome_limpo.title()

    def exibir_informacoes(self):
        """Retorna uma apresentação formatada dos dados do cliente."""
        return (f"Titular: {self.nome}\n"
                f"CPF: {self.cpf}\n"
                f"Telefone: {self.telefone}")


class ContaBancaria:
    """Gerencia o saldo, validações de senha, depósitos, saques e extratos."""
    def __init__(self, cliente, senha, saldo_inicial=0.0):
        self.cliente = cliente
        self.senha = self._validar_senha(senha)
        self.saldo = float(saldo_inicial)
        self.extrato = []

    def _validar_senha(self, senha):
        """Garante que a senha possua exatamente 4 dígitos numéricos."""
        senha_str = str(senha).strip()
        if not (senha_str.isdigit() and len(senha_str) == 4):
            raise ValueError("A senha cadastrada deve conter exatamente 4 dígitos numéricos.")
        return senha_str

    def verificar_senha(self, senha_informada):
        """Valida a senha informada pelo usuário na hora da operação."""
        return str(senha_informada).strip() == self.senha

    def depositar(self, valor):
        """Adiciona um valor ao saldo da conta."""
        if valor <= 0:
            return False, "O valor do depósito deve ser maior que R$ 0,00."
        self.saldo += valor
        self.extrato.append(f"Depósito: +R$ {valor:.2f}")
        return True, f"Depósito de R$ {valor:.2f} realizado com sucesso! Saldo atual: R$ {self.saldo:.2f}"

    def sacar(self, valor, senha_informada):
        """Realiza o saque de dinheiro mediante confirmação da senha."""
        if not self.verificar_senha(senha_informada):
            return False, "Senha incorreta! Operação de saque negada."
        if valor <= 0:
            return False, "O valor do saque deve ser maior que R$ 0,00."
        if valor > self.saldo:
            return False, "Recusa: Saldo insuficiente para realizar a operação."

        self.saldo -= valor
        self.extrato.append(f"Saque: -R$ {valor:.2f}")
        return True, f"Saque de R$ {valor:.2f} realizado com sucesso! Saldo atual: R$ {self.saldo:.2f}"

    def transferir(self, valor, conta_destino, senha_informada):
        """Transfere valor para outra conta com validação de senha."""
        if not self.verificar_senha(senha_informada):
            return False, "Senha incorreta! Operação de transferência negada."
        if valor <= 0:
            return False, "O valor da transferência deve ser maior que R$ 0,00."
        if valor > self.saldo:
            return False, "Transferência negada: Saldo insuficiente."
        if conta_destino == self:
            return False, "Não é possível transferir para a própria conta."

        self.saldo -= valor
        self.extrato.append(f"Transferência enviada: -R$ {valor:.2f} para {conta_destino.cliente.nome}")

        conta_destino.saldo += valor
        conta_destino.extrato.append(f"Transferência recebida: +R$ {valor:.2f} de {self.cliente.nome}")
        return True, f"Transferência de R$ {valor:.2f} para {conta_destino.cliente.nome} realizada com sucesso!"

    def obter_extrato(self):
        """Retorna o histórico das movimentações efetuadas na sessão."""
        if not self.extrato:
            return "Nenhuma movimentação registrada nesta sessão."
        return "\n".join(self.extrato)


class Banco:
    """Banco de dados dinâmico para gerenciamento central das contas registradas."""
    def __init__(self):
        self.contas = {} # Mapeamento por CPF: {cpf: objeto_conta}

    def cadastrar_cliente_e_conta(self, nome, cpf, telefone, senha, saldo_inicial=0.0):
        """Cadastra cliente e cria uma conta no sistema."""
        try:
            cpf_limpo = cpf.strip()
            if cpf_limpo in self.contas:
                return False, f"Já existe uma conta cadastrada para o CPF '{cpf_limpo}'."

            cliente = Cliente(nome, cpf_limpo, telefone)
            nova_conta = ContaBancaria(cliente, senha, saldo_inicial)
            self.contas[cpf_limpo] = nova_conta
            return True, f"Conta do cliente '{cliente.nome}' cadastrada com sucesso!"
        except ValueError as erro:
            return False, str(erro)

    def buscar_conta_por_cpf(self, cpf):
        """Retorna a conta cadastrada a partir do CPF informado."""
        return self.contas.get(cpf.strip(), None)


def menu_interativo():
    """Menu principal iterativo utilizando a função input()."""
    banco = Banco()

    while True:
        print("\n" + "=" * 45)
        print("         AMAZÔNIA BANK - MENU INTERATIVO")
        print("=" * 45)
        print("1. Cadastrar Novo Cliente e Conta")
        print("2. Depositar Dinheiro")
        print("3. Sacar Dinheiro (Exige Senha)")
        print("4. Transferir Dinheiro (Exige Senha)")
        print("5. Consultar Informações do Cliente")
        print("6. Ver Extrato da Conta")
        print("0. Sair")
        print("=" * 45)

        opcao = input("Escolha uma opção: ").strip()

        if opcao == "1":
            print("\n--- CADASTRO DE CLIENTE ---")
            nome = input("Digite o nome completo do titular: ")
            cpf = input("Digite o CPF do cliente: ")
            telefone = input("Digite o telefone do cliente: ")
            senha = input("Crie uma senha (4 dígitos numéricos): ")
            try:
                saldo_inicial = float(input("Digite o saldo inicial (ou 0): ") or "0")
            except ValueError:
                print("Valor inválido para o saldo inicial. Definido como R$ 0.00.")
                saldo_inicial = 0.0

            sucesso, msg = banco.cadastrar_cliente_e_conta(nome, cpf, telefone, senha, saldo_inicial)
            print(f"\nResultado: {msg}")

        elif opcao == "2":
            print("\n--- OPERAÇÃO DE DEPÓSITO ---")
            cpf = input("Digite o CPF da conta de destino: ")
            conta = banco.buscar_conta_por_cpf(cpf)
            if conta:
                try:
                    valor = float(input("Digite o valor do depósito: R$ "))
                    sucesso, msg = conta.depositar(valor)
                    print(f"\nResultado: {msg}")
                except ValueError:
                    print("Valor de depósito inválido.")
            else:
                print("Conta não encontrada para o CPF informado.")

        elif opcao == "3":
            print("\n--- OPERAÇÃO DE SAQUE ---")
            cpf = input("Digite o seu CPF: ")
            conta = banco.buscar_conta_por_cpf(cpf)
            if conta:
                try:
                    valor = float(input("Digite o valor do saque: R$ "))
                    senha = input("Digite a sua senha de 4 dígitos: ")
                    sucesso, msg = conta.sacar(valor, senha)
                    print(f"\nResultado: {msg}")
                except ValueError:
                    print("Valor de saque inválido.")
            else:
                print("Conta não encontrada para o CPF informado.")

        elif opcao == "4":
            print("\n--- OPERAÇÃO DE TRANSFERÊNCIA ---")
            cpf_origem = input("Digite o CPF da conta de origem: ")
            conta_origem = banco.buscar_conta_por_cpf(cpf_origem)
            if conta_origem:
                cpf_destino = input("Digite o CPF da conta de destino: ")
                conta_destino = banco.buscar_conta_por_cpf(cpf_destino)
                if conta_destino:
                    try:
                        valor = float(input("Digite o valor da transferência: R$ "))
                        senha = input("Digite a sua senha de 4 dígitos: ")
                        sucesso, msg = conta_origem.transferir(valor, conta_destino, senha)
                        print(f"\nResultado: {msg}")
                    except ValueError:
                        print("Valor de transferência inválido.")
                else:
                    print("Conta de destino não encontrada.")
            else:
                print("Conta de origem não encontrada.")

        elif opcao == "5":
            print("\n--- CONSULTA DE CLIENTE ---")
            cpf = input("Digite o CPF do cliente: ")
            conta = banco.buscar_conta_por_cpf(cpf)
            if conta:
                print("\nDados do Cliente:")
                print(conta.cliente.exibir_informacoes())
                print(f"Saldo Atual: R$ {conta.saldo:.2f}")
            else:
                print("Cliente não encontrado.")

        elif opcao == "6":
            print("\n--- EXTRATO DA CONTA ---")
            cpf = input("Digite o CPF do cliente: ")
            conta = banco.buscar_conta_por_cpf(cpf)
            if conta:
                print(f"\nExtrato de {conta.cliente.nome}:")
                print(conta.obter_extrato())
            else:
                print("Conta não encontrada.")

        elif opcao == "0":
            print("\nSessão encerrada com sucesso!")
            break

        else:
            print("\nOpção inválida! Tente novamente.")

if __name__ == "__main__":
    menu_interativo()


         AMAZÔNIA BANK - MENU INTERATIVO
1. Cadastrar Novo Cliente e Conta
2. Depositar Dinheiro
3. Sacar Dinheiro (Exige Senha)
4. Transferir Dinheiro (Exige Senha)
5. Consultar Informações do Cliente
6. Ver Extrato da Conta
0. Sair
Escolha uma opção: 1

--- CADASTRO DE CLIENTE ---
Digite o nome completo do titular: Mendes
Digite o CPF do cliente: 745736626
Digite o telefone do cliente: 6363462462
Crie uma senha (4 dígitos numéricos): 4245
Digite o saldo inicial (ou 0): 200

Resultado: Conta do cliente 'Mendes' cadastrada com sucesso!

         AMAZÔNIA BANK - MENU INTERATIVO
1. Cadastrar Novo Cliente e Conta
2. Depositar Dinheiro
3. Sacar Dinheiro (Exige Senha)
4. Transferir Dinheiro (Exige Senha)
5. Consultar Informações do Cliente
6. Ver Extrato da Conta
0. Sair
Escolha uma opção: 2

--- OPERAÇÃO DE DEPÓSITO ---
Digite o CPF da conta de destino: 745736626
Digite o valor do depósito: R$ 200000

Resultado: Depósito de R$ 200000.00 realizado com sucesso! Saldo atual: R$ 200200.00

  

In [7]:
# =====================================================================
# Script de Gerenciamento Bancário com Validação de CPF - Amazônia Bank
# Descrição: Sistema interativo via terminal com validação matemática
# rigorosa de CPF, senha de 4 dígitos, saldo inicial e movimentações.
# =====================================================================

def validar_cpf(cpf):
    """
    Função matemática para verificar se um CPF é válido.
    Remove caracteres não numéricos, verifica o tamanho, sequências
    repetidas e calcula os dois dígitos verificadores.
    """
    # Remove tudo o que não for dígito
    cpf_limpo = "".join([c for c in cpf if c.isdigit()])

    # Verifica se tem 11 dígitos
    if len(cpf_limpo) != 11:
        return False

    # Descarta CPFs com todos os dígitos iguais (ex: "111.111.111-11")
    if cpf_limpo == cpf_limpo[0] * 11:
        return False

    # --- Cálculo do 1º Dígito Verificador ---
    soma = sum(int(cpf_limpo[i]) * (10 - i) for i in range(9))
    resto = (soma * 10) % 11
    digito1 = 0 if resto == 10 else resto

    # --- Cálculo do 2º Dígito Verificador ---
    soma = sum(int(cpf_limpo[i]) * (11 - i) for i in range(10))
    resto = (soma * 10) % 11
    digito2 = 0 if resto == 10 else resto

    # Confirma se os dígitos calculados batem com os dígitos informados
    return digito1 == int(cpf_limpo[9]) and digito2 == int(cpf_limpo[10])


class Cliente:
    """Armazena e valida as informações pessoais do cliente."""
    def __init__(self, nome, cpf, telefone):
        self.nome = self._validar_nome(nome)
        self.cpf = self._validar_e_formatar_cpf(cpf)
        self.telefone = telefone.strip()

    def _validar_nome(self, nome):
        nome_limpo = nome.strip()
        if not nome_limpo or ";" in nome_limpo:
            raise ValueError("Nome de cliente inválido ou contém caractere proibido (;).")
        return nome_limpo.title()

    def _validar_e_formatar_cpf(self, cpf):
        if not validar_cpf(cpf):
            raise ValueError("CPF inválido! Verifique os números digitados.")
        # Retorna apenas os números limpos para padronização no dicionário
        return "".join([c for c in cpf if c.isdigit()])

    def exibir_informacoes(self):
        """Retorna uma apresentação formatada dos dados do cliente."""
        cpf_formatado = f"{self.cpf[:3]}.{self.cpf[3:6]}.{self.cpf[6:9]}-{self.cpf[9:]}"
        return (f"Titular: {self.nome}\n"
                f"CPF: {cpf_formatado}\n"
                f"Telefone: {self.telefone}")


class ContaBancaria:
    """Gerencia o saldo, validações de senha, depósitos, saques e extratos."""
    def __init__(self, cliente, senha, saldo_inicial=0.0):
        self.cliente = cliente
        self.senha = self._validar_senha(senha)
        self.saldo = float(saldo_inicial)
        self.extrato = []

    def _validar_senha(self, senha):
        """Garante que a senha possua exatamente 4 dígitos numéricos."""
        senha_str = str(senha).strip()
        if not (senha_str.isdigit() and len(senha_str) == 4):
            raise ValueError("A senha cadastrada deve conter exatamente 4 dígitos numéricos.")
        return senha_str

    def verificar_senha(self, senha_informada):
        """Valida a senha informada pelo usuário na hora da operação."""
        return str(senha_informada).strip() == self.senha

    def depositar(self, valor):
        """Adiciona um valor ao saldo da conta."""
        if valor <= 0:
            return False, "O valor do depósito deve ser maior que R$ 0,00."
        self.saldo += valor
        self.extrato.append(f"Depósito: +R$ {valor:.2f}")
        return True, f"Depósito de R$ {valor:.2f} realizado com sucesso! Saldo atual: R$ {self.saldo:.2f}"

    def sacar(self, valor, senha_informada):
        """Realiza o saque de dinheiro mediante confirmação da senha."""
        if not self.verificar_senha(senha_informada):
            return False, "Senha incorreta! Operação de saque negada."
        if valor <= 0:
            return False, "O valor do saque deve ser maior que R$ 0,00."
        if valor > self.saldo:
            return False, "Recusa: Saldo insuficiente para realizar a operação."

        self.saldo -= valor
        self.extrato.append(f"Saque: -R$ {valor:.2f}")
        return True, f"Saque de R$ {valor:.2f} realizado com sucesso! Saldo atual: R$ {self.saldo:.2f}"

    def transferir(self, valor, conta_destino, senha_informada):
        """Transfere valor para outra conta com validação de senha."""
        if not self.verificar_senha(senha_informada):
            return False, "Senha incorreta! Operação de transferência negada."
        if valor <= 0:
            return False, "O valor da transferência deve ser maior que R$ 0,00."
        if valor > self.saldo:
            return False, "Transferência negada: Saldo insuficiente."
        if conta_destino == self:
            return False, "Não é possível transferir para a própria conta."

        self.saldo -= valor
        self.extrato.append(f"Transferência enviada: -R$ {valor:.2f} para {conta_destino.cliente.nome}")

        conta_destino.saldo += valor
        conta_destino.extrato.append(f"Transferência recebida: +R$ {valor:.2f} de {self.cliente.nome}")
        return True, f"Transferência de R$ {valor:.2f} para {conta_destino.cliente.nome} realizada com sucesso!"

    def obter_extrato(self):
        """Retorna o histórico das movimentações efetuadas na sessão."""
        if not self.extrato:
            return "Nenhuma movimentação registrada nesta sessão."
        return "\n".join(self.extrato)


class Banco:
    """Banco de dados dinâmico para gerenciamento central das contas registradas."""
    def __init__(self):
        self.contas = {} # Mapeamento por CPF limpo: {cpf: objeto_conta}

    def cadastrar_cliente_e_conta(self, nome, cpf, telefone, senha, saldo_inicial=0.0):
        """Cadastra cliente e cria uma conta no sistema após validar o CPF."""
        try:
            # Limpa o CPF temporariamente para checar duplicidade antes de instanciar
            cpf_limpo = "".join([c for c in cpf if c.isdigit()])
            if cpf_limpo in self.contas:
                return False, f"Já existe uma conta cadastrada para este CPF."

            cliente = Cliente(nome, cpf, telefone)
            nova_conta = ContaBancaria(cliente, senha, saldo_inicial)
            self.contas[cliente.cpf] = nova_conta
            return True, f"Conta do cliente '{cliente.nome}' cadastrada com sucesso!"
        except ValueError as erro:
            return False, str(erro)

    def buscar_conta_por_cpf(self, cpf):
        """Retorna a conta cadastrada a partir do CPF informado."""
        cpf_limpo = "".join([c for c in cpf if c.isdigit()])
        return self.contas.get(cpf_limpo, None)


def menu_interativo():
    """Menu principal iterativo utilizando a função input()."""
    banco = Banco()

    while True:
        print("\n" + "=" * 45)
        print("         AMAZÔNIA BANK - MENU INTERATIVO")
        print("=" * 45)
        print("1. Cadastrar Novo Cliente e Conta")
        print("2. Depositar Dinheiro")
        print("3. Sacar Dinheiro (Exige Senha)")
        print("4. Transferir Dinheiro (Exige Senha)")
        print("5. Consultar Informações do Cliente")
        print("6. Ver Extrato da Conta")
        print("0. Sair")
        print("=" * 45)

        opcao = input("Escolha uma opção: ").strip()

        if opcao == "1":
            print("\n--- CADASTRO DE CLIENTE ---")
            nome = input("Digite o nome completo do titular: ")
            cpf = input("Digite o CPF do cliente (com ou sem pontos/traço): ")
            telefone = input("Digite o telefone do cliente: ")
            senha = input("Crie uma senha (4 dígitos numéricos): ")
            try:
                saldo_inicial = float(input("Digite o saldo inicial (ou 0): ") or "0")
            except ValueError:
                print("Valor inválido para o saldo inicial. Definido como R$ 0.00.")
                saldo_inicial = 0.0

            sucesso, msg = banco.cadastrar_cliente_e_conta(nome, cpf, telefone, senha, saldo_inicial)
            print(f"\nResultado: {msg}")

        elif opcao == "2":
            print("\n--- OPERAÇÃO DE DEPÓSITO ---")
            cpf = input("Digite o CPF da conta de destino: ")
            conta = banco.buscar_conta_por_cpf(cpf)
            if conta:
                try:
                    valor = float(input("Digite o valor do depósito: R$ "))
                    sucesso, msg = conta.depositar(valor)
                    print(f"\nResultado: {msg}")
                except ValueError:
                    print("Valor de depósito inválido.")
            else:
                print("Conta não encontrada para o CPF informado.")

        elif opcao == "3":
            print("\n--- OPERAÇÃO DE SAQUE ---")
            cpf = input("Digite o seu CPF: ")
            conta = banco.buscar_conta_por_cpf(cpf)
            if conta:
                try:
                    valor = float(input("Digite o valor do saque: R$ "))
                    senha = input("Digite a sua senha de 4 dígitos: ")
                    sucesso, msg = conta.sacar(valor, senha)
                    print(f"\nResultado: {msg}")
                except ValueError:
                    print("Valor de saque inválido.")
            else:
                print("Conta não encontrada para o CPF informado.")

        elif opcao == "4":
            print("\n--- OPERAÇÃO DE TRANSFERÊNCIA ---")
            cpf_origem = input("Digite o CPF da conta de origem: ")
            conta_origem = banco.buscar_conta_por_cpf(cpf_origem)
            if conta_origem:
                cpf_destino = input("Digite o CPF da conta de destino: ")
                conta_destino = banco.buscar_conta_por_cpf(cpf_destino)
                if conta_destino:
                    try:
                        valor = float(input("Digite o valor da transferência: R$ "))
                        senha = input("Digite a sua senha de 4 dígitos: ")
                        sucesso, msg = conta_origem.transferir(valor, conta_destino, senha)
                        print(f"\nResultado: {msg}")
                    except ValueError:
                        print("Valor de transferência inválido.")
                else:
                    print("Conta de destino não encontrada.")
            else:
                print("Conta de origem não encontrada.")

        elif opcao == "5":
            print("\n--- CONSULTA DE CLIENTE ---")
            cpf = input("Digite o CPF do cliente: ")
            conta = banco.buscar_conta_por_cpf(cpf)
            if conta:
                print("\nDados do Cliente:")
                print(conta.cliente.exibir_informacoes())
                print(f"Saldo Atual: R$ {conta.saldo:.2f}")
            else:
                print("Cliente não encontrado.")

        elif opcao == "6":
            print("\n--- EXTRATO DA CONTA ---")
            cpf = input("Digite o CPF do cliente: ")
            conta = banco.buscar_conta_por_cpf(cpf)
            if conta:
                print(f"\nExtrato de {conta.cliente.nome}:")
                print(conta.obter_extrato())
            else:
                print("Conta não encontrada.")

        elif opcao == "0":
            print("\nSessão encerrada com sucesso!")
            break

        else:
            print("\nOpção inválida! Tente novamente.")

if __name__ == "__main__":
    menu_interativo()


         AMAZÔNIA BANK - MENU INTERATIVO
1. Cadastrar Novo Cliente e Conta
2. Depositar Dinheiro
3. Sacar Dinheiro (Exige Senha)
4. Transferir Dinheiro (Exige Senha)
5. Consultar Informações do Cliente
6. Ver Extrato da Conta
0. Sair
Escolha uma opção: 1

--- CADASTRO DE CLIENTE ---
Digite o nome completo do titular: eori
Digite o CPF do cliente (com ou sem pontos/traço): 3532531535135
Digite o telefone do cliente: 3253262624
Crie uma senha (4 dígitos numéricos): 2362
Digite o saldo inicial (ou 0): 5

Resultado: CPF inválido! Verifique os números digitados.

         AMAZÔNIA BANK - MENU INTERATIVO
1. Cadastrar Novo Cliente e Conta
2. Depositar Dinheiro
3. Sacar Dinheiro (Exige Senha)
4. Transferir Dinheiro (Exige Senha)
5. Consultar Informações do Cliente
6. Ver Extrato da Conta
0. Sair
Escolha uma opção: 0

Sessão encerrada com sucesso!
